In [313]:
using LowLevelFEM, LinearAlgebra

In [314]:
p = 3

3

In [315]:
structured_rect_mesh(n=10, order=p)

mat = Material("body")
Θ = Field([mat], type=:ScalarField, dim=2, field=:T);

In [316]:
k = mat.k

s = ScalarField(Θ, "right", (x, y, z)->y > 0.5 ? 1 : 10)

K = ∫(Grad(Θ) ⋅ k ⋅ Grad(Θ))
q = ∫(Θ ⋅ s, Γ="right")

bc = BoundaryCondition("left", T=0);

In [317]:
fixed = constrainedDoFs(Θ, [bc])
free = freeDoFs(Θ, [bc]);

In [318]:
T1 = applyBoundaryConditions(Θ, [bc])
q_kin = K.A[:, fixed] * T1.a[fixed, 1]
T1.a[free] = (K.A[free, free]) \ (q.a[free, 1] - q_kin[free, 1]);

In [319]:
showDoFResults(T1, name="T1", visible=true);

In [320]:
T, R = reductionMatrices(Θ);

In [321]:
T2 = applyBoundaryConditions(Θ, [bc])
q_kin = K.A[:, fixed] * T2.a[fixed, 1]
Kr = T[free, :]' * K.A[free, free] * T[free, :]
qr = T[free, :]' * (q.a[free, 1] - q_kin[free, 1])

Tr = Kr \ qr

T2.a[free] = (T*Tr)[free];

In [322]:
showDoFResults(T2, name="T2", visible=true);

In [323]:
norm(T2.a - T1.a) / norm(T1.a)

0.00024174134892430234

In [324]:
∫(Θ, "body", T1)

0.062361111111114476

In [325]:
∫(Θ, "body", T2)

0.06236111111111806

In [326]:
openPostProcessor();